# 🎓 Visoria: Attention Detection & Explainable AI (XAI) Model  — v2 (audited & optimized)
### An Attention-Aware Socratic Pedagogical Intervention System
**Researchers:** Omor Faruk (2202026), Mostafa Shariare (2202045)  
**Supervisor:** Farhana Islam, Assistant Professor  
**Affiliation:** Department of Educational Technology and Engineering, University of Frontier Technology, Bangladesh

---
### 📌 What changed vs. the original notebook
| # | Problem in original | Fix in this version |
|---|---|---|
| 1 | **Class names inverted.** In the real CSV, `label = 1` means **Distracted** (phone → label 1 in 100% of rows; the "forward + no phone" rule scored 4.9% accuracy). The notebook called `1` "Attentive" everywhere (plots, reports, LIME text, `attention_score_pct`, intervention flag). | Label semantics are **verified from the data**, then mapped explicitly. `attention_score_pct` is now `P(Attentive)`. |
| 2 | **Crash:** `CalibratedClassifierCV(cv="prefit")` was removed in scikit-learn 1.8 (and deprecated from 1.6). Cells 14–15 then failed with `final_model` undefined. | Calibrated with `cv=5, ensemble=False` — works on all recent versions. |
| 3 | **Calibration on training data** (`prefit` + `.fit(X_train)`) — calibrating on the data the model was trained on is invalid. | Calibrator is trained on **out-of-fold** predictions. Method (sigmoid vs isotonic) is chosen by CV on train only. |
| 4 | **Silent synthetic-data fallback** with the *opposite* label meaning — a missing CSV would have produced fake metrics without warning. | Removed. Missing file → clear error. |
| 5 | **Redundant / useless features:** `face_h` ≈ `face_w` (corr = 1.0000), so `face_aspect_ratio` is constant noise (std ≈ 3e-5); `face_area` duplicates `face_w`. | Dropped. Phone bounding-box coordinates dropped (position adds nothing; presence + confidence kept). |
| 6 | `StandardScaler` used with tree models (no effect) but added latency + one more artifact that can drift. | Removed. |
| 7 | **Ablation compared different models** (tuned XGBoost vs. default Random Forest). | Ablation refits the *same* tuned estimator for every variant. |
| 8 | SHAP cell picked `tuned_model.estimator`, which for `RandomForestClassifier` is an **unfitted** tree template → silently caught by `try/except`. | SHAP uses the fitted base estimator explicitly; added a **single-row SHAP** function with latency measurement. |
| 9 | Selection only by accuracy/F1; no latency, size, or interval estimates. | Benchmark reports F1, AUC, Brier, **inference ms, model size**; final pick uses an efficiency-aware rule; bootstrap 95 % CIs. |
| 10 | Duplicates (30 rows) left in → can appear in both train and test. | De-duplicated before splitting. |
| 11 | Training-time and serving-time feature code were separate copies (parity risk); serving used `pd.get_dummies` per frame (~3 ms). | One `attention_features.py` shared by training and backend, with a pure-Python single-frame fast path (parity-tested against the batch path). |

⚠️ **Read the "label circularity" cell (§4b).** In this dataset the label is almost a deterministic function of pose, phone, and face count, so near-100 % scores are expected and should be framed carefully in the paper.

In [ ]:
# 1. Install required packages (Google Colab)
!pip install -q lightgbm xgboost shap lime joblib scikit-learn matplotlib seaborn

In [ ]:
# 2. Imports, config, plot style
import os, sys, json, time, pickle, warnings, importlib
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_validate, RandomizedSearchCV)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             matthews_corrcoef, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay, brier_score_loss)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap, joblib
from lime.lime_tabular import LimeTabularExplainer

plt.rcParams.update({"figure.dpi": 120, "font.size": 10.5, "axes.titlesize": 12.5, "axes.labelsize": 11,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATASET_FILENAME = "attention_detection_dataset_v1.csv"
ARTIFACTS_DIR = Path("artifacts"); ARTIFACTS_DIR.mkdir(exist_ok=True)
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

FORCE_MODEL = None            # e.g. "LightGBM" to override the automatic pick in §6
INTERVENTION_THRESHOLD = 0.60 # intervene when P(Attentive) is below this
COLORS = {"Attentive": "#22C55E", "Distracted": "#EF4444"}

print(f"✓ Ready | scikit-learn {sklearn.__version__} | numpy {np.__version__} | pandas {pd.__version__}")

## 📂 3. Data Ingestion
Upload `attention_detection_dataset_v1.csv` to Colab (left sidebar → Files) or run the cell and use the upload prompt.
There is intentionally **no synthetic fallback**: reporting metrics on generated data by accident would invalidate the study.

In [ ]:
# 3. Load dataset (strict — fails loudly if missing)
candidates = [DATASET_FILENAME, f"data/{DATASET_FILENAME}", f"ml/data/{DATASET_FILENAME}",
              f"../data/{DATASET_FILENAME}", f"../ml/data/{DATASET_FILENAME}", f"/content/{DATASET_FILENAME}"]
found = next((p for p in candidates if os.path.exists(p)), None)

if found is None:
    try:
        from google.colab import files
        print(f"Select '{DATASET_FILENAME}' to upload:")
        up = files.upload()
        if DATASET_FILENAME in up:
            found = DATASET_FILENAME
    except ImportError:
        pass

if found is None:
    raise FileNotFoundError(f"'{DATASET_FILENAME}' not found. Upload it and re-run this cell.")

df_raw = pd.read_csv(found)
print(f"✓ Loaded {found}: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head()

## 🧾 4. Data Audit
Checks schema, duplicates, missing values, **what `label` actually means**, redundant columns, and sentinel values.

In [ ]:
# 4. Data audit
EXPECTED = ["no_of_face","face_x","face_y","face_w","face_h","face_con","no_of_hand","pose","pose_x","pose_y",
            "phone","phone_x","phone_y","phone_w","phone_h","phone_con","label"]
missing_cols = [c for c in EXPECTED if c not in df_raw.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"

print("Missing values:", int(df_raw.isna().sum().sum()))
n_dup = int(df_raw.duplicated().sum())
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {n_dup}  →  {len(df)} rows remain")

# --- What does label=1 mean? Use the data, not assumptions. ---
phone_rate = df.groupby("label")["phone"].mean()
print("\nPhone-detected rate by raw label:\n", phone_rate.round(4).to_string())
DISTRACTED_RAW = int(phone_rate.idxmax())          # class that co-occurs with phones
print(f"\n→ Raw label {DISTRACTED_RAW} = DISTRACTED,  raw label {1-DISTRACTED_RAW} = ATTENTIVE")

# Internal target convention used everywhere below:  y = 1 → Distracted,  y = 0 → Attentive
y_all = (df["label"] == DISTRACTED_RAW).astype(int)
CLASS_NAMES = ["Attentive", "Distracted"]
print("\nClass balance:", y_all.map(dict(enumerate(CLASS_NAMES))).value_counts().to_dict())

# --- Redundancy & sentinels ---
ar = (df.face_w / (df.face_h + 1e-6))[df.face_h > 0]
print(f"\ncorr(face_w, face_h) = {df.face_w.corr(df.face_h):.6f}; face aspect ratio std = {ar.std():.2e}  → face_h & aspect ratio carry no information")
sent = ((df.pose_x == -15) & (df.pose_y == 10)).sum()
print(f"Rows with pose_x=-15 & pose_y=10 (constant sentinel): {sent}  (all pose='down': {df.loc[(df.pose_x==-15)&(df.pose_y==10),'pose'].eq('down').all()})")
print(f"Rows with no face detected (all face_* = 0): {(df.no_of_face==0).sum()}")

In [ ]:
# 5. EDA (class names corrected)
state = y_all.map(dict(enumerate(CLASS_NAMES)))
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.countplot(x=state, order=CLASS_NAMES, hue=state, palette=COLORS, legend=False, ax=axes[0])
axes[0].set_title("Class balance"); axes[0].set_xlabel("")
sns.countplot(x=df["pose"], hue=state, hue_order=CLASS_NAMES, palette=COLORS, ax=axes[1])
axes[1].set_title("Head pose vs. state"); axes[1].legend(title="")
sns.countplot(x=df["phone"].map({0:"No phone",1:"Phone"}), hue=state, hue_order=CLASS_NAMES, palette=COLORS, ax=axes[2])
axes[2].set_title("Phone vs. state"); axes[2].set_xlabel(""); axes[2].legend(title="")
plt.tight_layout(); plt.show()

## 🔍 4b. Rule Baseline & Label-Circularity Check
Two questions: (1) how far does a trivial rule go, and (2) **how much of the label is explained by just pose + phone + face count?**
If the answer is "almost all", high ML scores mostly show the model can *reconstruct the labeling rule* — reviewers will ask about this.

In [ ]:
# 6. Corrected rule baseline + circularity check
rule_pred = ((df.pose != "forward") | (df.phone == 1) | (df.no_of_face != 1)).astype(int)   # 1 = distracted
print("Trivial rule  'distracted if pose≠forward OR phone OR face≠1':")
print(f"  Accuracy {accuracy_score(y_all, rule_pred):.4f} | Macro-F1 {f1_score(y_all, rule_pred, average='macro'):.4f}")

rule_feats = pd.DataFrame({
    "no_of_face": df.no_of_face, "phone": df.phone, "pose_x": df.pose_x, "pose_y": df.pose_y,
    **{f"pose_{p}": (df.pose == p).astype(int) for p in ["down","forward","left","right"]}})
cv_quick = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
for depth in (3, 5, 8):
    s = cross_validate(DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE), rule_feats, y_all,
                       cv=cv_quick, scoring="f1_macro")
    print(f"  Depth-{depth} tree using ONLY pose/phone/face-count features: CV macro-F1 = {s['test_score'].mean():.4f}")

resid = df[(df.pose == "forward") & (df.phone == 0) & (df.no_of_face == 1)]
print(f"\nResidual subset (forward, no phone, 1 face): {len(resid)} rows, {int(y_all[resid.index].sum())} distracted")
t = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(resid[["pose_x","pose_y"]], y_all[resid.index])
print(export_text(t, feature_names=["pose_x","pose_y"]))
print("→ Even the 'hard' cases are separated by head-angle thresholds. Report this openly and lean on the ablation (§10).")

## ⚙️ 5. Feature Engineering & Split
A single feature module (`attention_features.py`) is written to disk and imported here **and** by the backend, so training and serving cannot drift apart.
No scaler is needed (all candidate models except the LR baseline are scale-invariant; LR uses its own internal pipeline).

In [ ]:
# 7. Shared feature module + stratified split
FEATURES_SRC = '''\
# Shared feature builder for the Visoria attention model (training + serving).
import pandas as pd

POSES = ("down", "forward", "left", "right")
NUMERIC = ["no_of_face", "face_x", "face_y", "face_w", "face_con", "no_of_hand",
           "pose_x", "pose_y", "phone", "phone_con"]
FEATURE_NAMES = NUMERIC + [f"pose_{p}" for p in POSES]


def build_features(raw) -> pd.DataFrame:
    # raw: dict (one frame) or DataFrame of raw telemetry -> float DataFrame in FEATURE_NAMES order.
    # Missing numeric fields become 0; unknown pose strings become an all-zero one-hot.
    d = pd.DataFrame([raw]) if isinstance(raw, dict) else raw
    out = pd.DataFrame(index=d.index)
    for c in NUMERIC:
        out[c] = pd.to_numeric(d[c], errors="coerce") if c in d.columns else 0.0
    out = out.fillna(0.0)
    pose = d["pose"].astype(str).str.lower() if "pose" in d.columns else pd.Series("", index=d.index)
    for p in POSES:
        out[f"pose_{p}"] = (pose == p).astype(float)
    return out[FEATURE_NAMES].astype(float)


def _num(v):
    try:
        x = float(v)
    except (TypeError, ValueError):
        return 0.0
    return 0.0 if x != x else x


def features_row(raw: dict) -> list:
    # Fast single-frame path (no pandas): same output as build_features(raw).iloc[0]
    v = [_num(raw.get(c, 0)) for c in NUMERIC]
    p = str(raw.get("pose", "")).lower()
    return v + [1.0 if p == q else 0.0 for q in POSES]
'''
(ARTIFACTS_DIR / "attention_features.py").write_text(FEATURES_SRC)
if str(ARTIFACTS_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(ARTIFACTS_DIR.resolve()))
import attention_features; importlib.reload(attention_features)
from attention_features import build_features, features_row, FEATURE_NAMES

X_all = build_features(df)
print(f"✓ {len(FEATURE_NAMES)} features:", FEATURE_NAMES)

# Serving fast-path must equal the batch path exactly
_chk = df.sample(300, random_state=RANDOM_STATE)
assert np.allclose(np.array([features_row(r) for r in _chk.to_dict("records")]), build_features(_chk).values)
print("✓ features_row() (serving) == build_features() (training) on 300 random rows")

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all)
print(f"✓ Train {len(X_train)} | Test {len(X_test)} (stratified 80/20, duplicates removed)")
print("ℹ️ The CSV has no subject/session id, so a group-aware split is impossible; consecutive video frames may be correlated. State this limitation in the paper.")

## 🏆 6. Efficiency-Aware Model Benchmark
Repeated stratified 5-fold CV (2 repeats) on the **training set only**. For each candidate we record accuracy, macro-F1, ROC-AUC, Brier score,
**single-row inference latency**, and **serialized size** — the numbers that matter for a real-time dashboard.

**Selection rule:** take every model whose macro-F1 is within one standard deviation (across folds) of the best, then pick the fastest / smallest of that group.

In [ ]:
# 8. Benchmark: quality + latency + size
def make_candidates():
    return {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=2, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=100, min_samples_leaf=2, n_jobs=1, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, n_jobs=1, verbosity=0, random_state=RANDOM_STATE),
        "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=15, max_depth=5, min_child_samples=10,
                                   n_jobs=1, verbose=-1, random_state=RANDOM_STATE),
    }

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
scoring = {"acc": "accuracy", "f1m": "f1_macro", "auc": "roc_auc", "brier": "neg_brier_score"}

def latency_ms(model, row, n=200):
    model.predict_proba(row)                      # warm-up
    t0 = time.perf_counter()
    for _ in range(n): model.predict_proba(row)
    return (time.perf_counter() - t0) / n * 1000

rows = []
row0 = X_train.iloc[[0]]
for name, est in make_candidates().items():
    s = cross_validate(est, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    fitted = clone(est).fit(X_train, y_train)
    rows.append({"Model": name, "Accuracy": s["test_acc"].mean(), "Macro-F1": s["test_f1m"].mean(), "F1-SD": s["test_f1m"].std(),
                 "ROC-AUC": s["test_auc"].mean(), "Brier": -s["test_brier"].mean(),
                 "ms/pred": latency_ms(fitted, row0), "Size (KB)": len(pickle.dumps(fitted)) / 1024})
bench = pd.DataFrame(rows).set_index("Model").sort_values("Macro-F1", ascending=False)

best = bench["Macro-F1"].idxmax()
tied = bench[bench["Macro-F1"] >= bench.loc[best, "Macro-F1"] - bench.loc[best, "F1-SD"]]
auto_pick = tied.sort_values(["ms/pred", "Size (KB)"]).index[0]
chosen_name = FORCE_MODEL or auto_pick

display(bench.style.format({"Accuracy": "{:.4f}", "Macro-F1": "{:.4f}", "F1-SD": "{:.4f}", "ROC-AUC": "{:.4f}",
                            "Brier": "{:.4f}", "ms/pred": "{:.2f}", "Size (KB)": "{:.1f}"}).highlight_max(subset=["Macro-F1","ROC-AUC"], color="#dcfce7")
        .highlight_min(subset=["Brier","ms/pred","Size (KB)"], color="#dcfce7"))
print(f"Statistically tied with best (within 1 SD): {list(tied.index)}")
why = "manual override" if FORCE_MODEL else ("only model within 1 SD of the best" if len(tied) == 1 else "fastest/smallest of the tied group")
print(f"✓ Selected: {chosen_name}  ({why})")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
b = bench.reset_index()
ax[0].bar(b["Model"], b["Macro-F1"], yerr=b["F1-SD"], capsize=4, color="#3B82F6"); ax[0].set_ylim(b["Macro-F1"].min()-0.02, 1.003)
ax[0].set_title("Macro-F1 (5×2 CV, ±1 SD)"); ax[0].tick_params(axis="x", rotation=25)
ax[1].scatter(b["ms/pred"], b["Macro-F1"], s=b["Size (KB)"].clip(20, 800), color="#8B5CF6", alpha=.7)
for _, r in b.iterrows(): ax[1].annotate(r["Model"], (r["ms/pred"], r["Macro-F1"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax[1].set_xlabel("Single-row latency (ms)"); ax[1].set_ylabel("Macro-F1"); ax[1].set_title("Accuracy vs. latency (bubble = model size)")
plt.tight_layout(); plt.show()

## 🎯 7. Hyperparameter Tuning
Randomized search (macro-F1) on the selected model only, with small-model search spaces so the tuned model stays fast.

In [ ]:
# 9. Tuning
PARAM_SPACES = {
    "Decision Tree": {"max_depth": [4,5,6,7,8,10,12], "min_samples_leaf": [1,2,3,5,8], "criterion": ["gini","entropy"],
                      "ccp_alpha": [0, 1e-4, 5e-4, 1e-3]},
    "LightGBM": {"n_estimators": [60,100,150,200], "num_leaves": [7,15,31], "max_depth": [3,4,5,6],
                 "learning_rate": [0.05,0.1,0.2], "min_child_samples": [5,10,20], "subsample": [0.8,1.0], "colsample_bytree": [0.8,1.0]},
    "XGBoost": {"n_estimators": [60,100,150,200], "max_depth": [3,4,5,6], "learning_rate": [0.05,0.1,0.2],
                "subsample": [0.8,1.0], "colsample_bytree": [0.8,1.0]},
    "Random Forest": {"n_estimators": [50,100,200], "max_depth": [8,12,16,None], "min_samples_leaf": [1,2,4],
                      "max_features": ["sqrt", 0.5]},
    "Logistic Regression": {},
}
space = PARAM_SPACES[chosen_name]
base_est = make_candidates()[chosen_name]

if space:
    n_combos = int(np.prod([len(v) for v in space.values()]))
    search = RandomizedSearchCV(base_est, space, n_iter=min(30, n_combos), cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                                scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    tuned = search.best_estimator_
    print(f"✓ Best CV macro-F1: {search.best_score_:.4f}\n✓ Best params: {search.best_params_}")
else:
    tuned = clone(base_est).fit(X_train, y_train)
print(f"Tuned model: {chosen_name} | {latency_ms(tuned, row0):.2f} ms/pred | {len(pickle.dumps(tuned))/1024:.1f} KB")

## 🧪 8. Held-Out Test Evaluation (with bootstrap 95 % CIs)
Class convention: **positive class = Distracted**.

In [ ]:
# 10. Test evaluation
y_pred = tuned.predict(X_test)
y_prob = tuned.predict_proba(X_test)[:, 1]          # P(Distracted)

def metrics(y_true, y_hat, y_p):
    return {"Accuracy": accuracy_score(y_true, y_hat), "Precision (Distracted)": precision_score(y_true, y_hat),
            "Recall (Distracted)": recall_score(y_true, y_hat), "F1 (Distracted)": f1_score(y_true, y_hat),
            "Macro-F1": f1_score(y_true, y_hat, average="macro"), "ROC-AUC": roc_auc_score(y_true, y_p),
            "MCC": matthews_corrcoef(y_true, y_hat)}

m_test = metrics(y_test, y_pred, y_prob)

rng = np.random.default_rng(RANDOM_STATE)
yt, yp = y_test.values, np.asarray(y_pred)
boots = {"Accuracy": [], "Macro-F1": []}
for _ in range(1000):
    i = rng.integers(0, len(yt), len(yt))
    boots["Accuracy"].append(accuracy_score(yt[i], yp[i])); boots["Macro-F1"].append(f1_score(yt[i], yp[i], average="macro"))
ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in boots.items()}

print("=" * 52, "\n  HELD-OUT TEST METRICS  (positive = Distracted)\n" + "=" * 52)
for k, v in m_test.items():
    extra = f"   95% CI [{ci[k][0]:.4f}, {ci[k][1]:.4f}]" if k in ci else ""
    print(f"{k:<24}: {v:.4f}{extra}")
print("\n", classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=CLASS_NAMES).plot(ax=axes[0], cmap="Blues", values_format="d", colorbar=False)
axes[0].set_title("Held-out confusion matrix"); axes[0].grid(False)
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], color="#22C55E", name=chosen_name)
axes[1].plot([0,1],[0,1],"k--",alpha=.5,label="Chance"); axes[1].set_title(f"ROC (AUC = {m_test['ROC-AUC']:.4f})"); axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

## ⚖️ 9. Probability Calibration (done correctly)
The dashboard shows a confidence %, so probabilities must be reliable. The calibrator is fit on **out-of-fold** predictions (`cv=5, ensemble=False` → one final model, tiny runtime cost). Sigmoid vs. isotonic is chosen by CV Brier on the **training set only**.

In [ ]:
# 11. Calibration
def ece_score(y_true, p, bins=10):
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1); e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any(): e += m.mean() * abs(np.asarray(y_true)[m].mean() - p[m].mean())
    return e

cal_cv = {}
for meth in ("sigmoid", "isotonic"):
    s = cross_validate(CalibratedClassifierCV(clone(tuned), method=meth, cv=5, ensemble=False), X_train, y_train,
                       cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="neg_brier_score", n_jobs=-1)
    cal_cv[meth] = -s["test_score"].mean()
CAL_METHOD = min(cal_cv, key=cal_cv.get)
print("CV Brier on train (lower is better):", {k: round(v, 5) for k, v in cal_cv.items()}, "→ using", CAL_METHOD)

final_model = CalibratedClassifierCV(clone(tuned), method=CAL_METHOD, cv=5, ensemble=False).fit(X_train, y_train)
base_model  = final_model.calibrated_classifiers_[0].estimator      # fitted tree/booster used for SHAP
y_prob_cal  = final_model.predict_proba(X_test)[:, 1]

rep = pd.DataFrame({"Brier": [brier_score_loss(y_test, y_prob), brier_score_loss(y_test, y_prob_cal)],
                    "ECE": [ece_score(y_test, y_prob), ece_score(y_test, y_prob_cal)]}, index=["Uncalibrated", f"Calibrated ({CAL_METHOD})"])
display(rep.round(5))

levels = len(np.unique(y_prob_cal.round(4))); mid = float(((y_prob_cal > 0.05) & (y_prob_cal < 0.95)).mean())
print(f"Score resolution on test: {levels} distinct calibrated values | {mid:.1%} of frames fall in the 5–95 % 'uncertain' band")
if levels < 10:
    print("⚠️ This model's confidence is effectively a hard flag (very few distinct scores). If the dashboard needs a graded attention score / a meaningful 0.60 threshold, set FORCE_MODEL = 'LightGBM' in §2 and re-run — accuracy on this data is statistically equivalent.")

pt_r, pp_r = calibration_curve(y_test, y_prob, n_bins=10, strategy="quantile")
pt_c, pp_c = calibration_curve(y_test, y_prob_cal, n_bins=10, strategy="quantile")
plt.figure(figsize=(5.8, 5)); plt.plot([0,1],[0,1],"k--",alpha=.5,label="Perfect")
plt.plot(pp_r, pt_r, "s-", color="#EF4444", label="Uncalibrated"); plt.plot(pp_c, pt_c, "o-", color="#22C55E", label=f"Calibrated ({CAL_METHOD})")
plt.xlabel("Predicted P(Distracted)"); plt.ylabel("Observed fraction distracted"); plt.title("Reliability diagram"); plt.legend(); plt.tight_layout(); plt.show()

## 🔬 10. Ablation Study (same tuned estimator for every variant)
How much do the dominant cues (phone, pose) matter — and do the remaining signals carry independent information?

In [ ]:
# 12. Ablation with identical model family/hyper-parameters
pose_cols  = ["pose_x", "pose_y"] + [c for c in FEATURE_NAMES if c.startswith("pose_") and c not in ("pose_x", "pose_y")]
phone_cols = ["phone", "phone_con"]
variants = {"Full model": [], "− phone": phone_cols, "− pose": pose_cols, "− pose & phone": pose_cols + phone_cols,
            "− face count": ["no_of_face"]}
abl = []
for name, drop in variants.items():
    cols = [c for c in FEATURE_NAMES if c not in drop]
    m = clone(tuned).fit(X_train[cols], y_train)
    p = m.predict(X_test[cols])
    abl.append({"Variant": name, "Features": len(cols), "Accuracy": accuracy_score(y_test, p), "Macro-F1": f1_score(y_test, p, average="macro")})
abl_df = pd.DataFrame(abl).set_index("Variant"); display(abl_df.round(4))

plt.figure(figsize=(7.5, 4)); bars = plt.bar(abl_df.index, abl_df["Macro-F1"], color=["#22C55E"] + ["#F59E0B"] * (len(abl_df) - 1), width=.55)
for b_, v in zip(bars, abl_df["Macro-F1"]): plt.text(b_.get_x() + b_.get_width()/2, v + .01, f"{v:.3f}", ha="center", fontweight="bold")
plt.ylim(0, 1.1); plt.ylabel("Test macro-F1"); plt.title("Ablation: contribution of feature groups"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 🧠 11. Explainable AI: SHAP (global + real-time single-row) and LIME
SHAP explains the fitted base model's raw output toward *Distracted* (probability units for a single tree, log-odds for boosted models; calibration is a monotonic step applied afterwards). Positive SHAP → pushes toward *Distracted*.

In [ ]:
# 13. SHAP — global + single-row
explainer = shap.TreeExplainer(base_model)

def shap_distracted(vals):
    if isinstance(vals, list): return np.asarray(vals[1])
    vals = np.asarray(vals)
    return vals[:, :, 1] if vals.ndim == 3 else vals

sample = X_test.sample(min(300, len(X_test)), random_state=RANDOM_STATE)
sv = shap_distracted(explainer.shap_values(sample))
plt.figure(figsize=(9, 5.5))
shap.summary_plot(sv, sample, feature_names=FEATURE_NAMES, show=False)
plt.title("SHAP global importance — push toward Distracted", fontsize=12.5, pad=12); plt.tight_layout(); plt.show()

def explain_row(raw_or_X, top_k=5):
    # Real-time SHAP for ONE frame -> ranked contributions (usable to ground LLM feedback).
    X = build_features(raw_or_X) if isinstance(raw_or_X, dict) else raw_or_X.iloc[[0]]
    v = shap_distracted(explainer.shap_values(X))[0]
    order = np.argsort(-np.abs(v))[:top_k]
    return [{"feature": FEATURE_NAMES[i], "value": float(X.iloc[0, i]), "shap": float(v[i]),
             "pushes_toward": "Distracted" if v[i] > 0 else "Attentive"} for i in order]

explain_row(X_test.iloc[[0]])                        # warm-up
t0 = time.perf_counter()
for i in range(50): explain_row(X_test.iloc[[i]])
shap_ms = (time.perf_counter() - t0) / 50 * 1000
print(f"Single-row SHAP latency: {shap_ms:.2f} ms/frame")
print(json.dumps(explain_row(X_test.iloc[[0]]), indent=2))

In [ ]:
# 14. LIME — local explanations (predict_fn keeps feature names)
def predict_fn(arr): return final_model.predict_proba(pd.DataFrame(arr, columns=FEATURE_NAMES))

lime_explainer = LimeTabularExplainer(X_train.values, feature_names=FEATURE_NAMES, class_names=CLASS_NAMES,
                                      mode="classification", random_state=RANDOM_STATE)
idx_att  = int(np.where(y_test.values == 0)[0][0])
idx_dist = int(np.where(y_test.values == 1)[0][0])

for title, idx in (("ATTENTIVE", idx_att), ("DISTRACTED", idx_dist)):
    e = lime_explainer.explain_instance(X_test.values[idx], predict_fn, num_features=6, labels=(1,))
    print(f"\nLIME — true {title} sample (P(Distracted) = {predict_fn(X_test.values[[idx]])[0,1]:.3f})")
    for feat, w in e.as_list(label=1):
        print(f"  {feat:<38}: {w:+.4f}  → {'toward Distracted' if w > 0 else 'toward Attentive'}")

## 💾 12. Export Artifacts
Bundle: calibrated model, column list, shared feature module, and metadata (class mapping, threshold, library versions). **No scaler** — not needed.

In [ ]:
# 15. Save artifacts
joblib.dump(final_model, ARTIFACTS_DIR / "attention_model.pkl", compress=3)
joblib.dump(FEATURE_NAMES, ARTIFACTS_DIR / "attention_columns.pkl")

meta = {
    "model_architecture": chosen_name, "calibration": CAL_METHOD,
    "class_names": CLASS_NAMES, "positive_class_index": 1, "positive_class": "Distracted",
    "raw_label_distracted": DISTRACTED_RAW, "intervention_threshold_p_attentive": INTERVENTION_THRESHOLD,
    "feature_names": FEATURE_NAMES, "sklearn_version": sklearn.__version__, "python": sys.version.split()[0],
}
(ARTIFACTS_DIR / "attention_meta.json").write_text(json.dumps(meta, indent=2))

summary = {**{f"test_{k.lower().replace(' ', '_').replace('(', '').replace(')', '')}": float(v) for k, v in m_test.items()},
           "accuracy_ci95": [float(x) for x in ci["Accuracy"]], "macro_f1_ci95": [float(x) for x in ci["Macro-F1"]],
           "brier_uncalibrated": float(rep.iloc[0, 0]), "brier_calibrated": float(rep.iloc[1, 0]),
           "ece_uncalibrated": float(rep.iloc[0, 1]), "ece_calibrated": float(rep.iloc[1, 1]),
           "single_row_shap_ms": float(shap_ms), "model": meta["model_architecture"],
           "ablation_macro_f1": {k: float(v) for k, v in abl_df["Macro-F1"].items()}}
(OUT_DIR / "visoria_model_summary.json").write_text(json.dumps(summary, indent=2))
bench.round(5).to_csv(OUT_DIR / "benchmark_table.csv")

import shutil
zip_path = shutil.make_archive("visoria_model_artifacts", "zip", ARTIFACTS_DIR)
print("✓ Artifacts:", sorted(p.name for p in ARTIFACTS_DIR.glob("*") if p.is_file()), f"| zip: {zip_path}")
print(json.dumps(summary, indent=2))
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    print("Not in Colab — grab the zip from the working directory.")

## 🚀 13. Real-Time Inference (loads artifacts from disk = exactly what the backend will do)

In [ ]:
# 16. Production-style predictor + parity and latency checks
class AttentionPredictor:
    def __init__(self, artifacts_dir="artifacts"):
        d = Path(artifacts_dir)
        if str(d.resolve()) not in sys.path: sys.path.insert(0, str(d.resolve()))
        from attention_features import features_row as _row
        self._row = _row
        self.model = joblib.load(d / "attention_model.pkl")
        self.cols = joblib.load(d / "attention_columns.pkl")
        self.meta = json.loads((d / "attention_meta.json").read_text())
        assert list(self.cols) == list(FEATURE_NAMES), "column order mismatch"
        self._att = list(self.model.classes_).index(0)                # column of P(Attentive)

    def predict(self, telemetry: dict) -> dict:
        X = pd.DataFrame([self._row(telemetry)], columns=self.cols)
        p_att = float(self.model.predict_proba(X)[0, self._att])
        return {"status": "Attentive" if p_att >= 0.5 else "Distracted",
                "attention_score_pct": round(p_att * 100, 1),
                "pedagogical_intervention_needed": p_att < self.meta["intervention_threshold_p_attentive"]}

predictor = AttentionPredictor(ARTIFACTS_DIR)

# Parity: reloaded model must reproduce in-memory predictions exactly
assert np.allclose(predictor.model.predict_proba(X_test[predictor.cols]), final_model.predict_proba(X_test), atol=1e-12)
print("✓ Reloaded artifact matches in-memory model")

learner_focused = {"no_of_face": 1, "face_x": 250, "face_y": 140, "face_w": 165, "face_h": 165, "face_con": 91.2,
                   "no_of_hand": 0, "pose": "forward", "pose_x": 0.2, "pose_y": 0.1,
                   "phone": 0, "phone_x": 0, "phone_y": 0, "phone_w": 0, "phone_h": 0, "phone_con": 0}
learner_distracted = {"no_of_face": 1, "face_x": 256, "face_y": 246, "face_w": 141, "face_h": 141, "face_con": 89.0,
                      "no_of_hand": 2, "pose": "down", "pose_x": 32.5, "pose_y": 36.5,
                      "phone": 1, "phone_x": 117, "phone_y": 137, "phone_w": 276, "phone_h": 421, "phone_con": 0.88}
print("Case 1 (focused):   ", json.dumps(predictor.predict(learner_focused)))
print("Case 2 (on phone):  ", json.dumps(predictor.predict(learner_distracted)))
print("Case 3 (empty dict):", json.dumps(predictor.predict({})), " ← graceful on missing telemetry")

ts = []
for _ in range(300):
    t0 = time.perf_counter(); predictor.predict(learner_focused); ts.append((time.perf_counter() - t0) * 1000)
print(f"\nEnd-to-end latency per frame (features + calibrated model): p50 {np.percentile(ts, 50):.2f} ms | p95 {np.percentile(ts, 95):.2f} ms")